# Notebook 09 — Paper Figures 09 / 10 / 11 (publication-quality)

**Goal:** produce the three named "headline figures" — fig 09 (scatter), fig 10 (per-component breakdown), fig 11 (freezing) — that go directly into the paper. Each saved as both PDF (for the manuscript) and PNG (for slides), 300 dpi, serif fonts.

**Why this notebook is small and surgical:** these three figures will be referenced by figure-number in the paper text. They must look exactly right (font, color, axis, annotation) and must be reproducible from the cleaned `master_wide.csv`. We do not invent any new analysis here — everything is a styled version of what notebook 08 already established.

**Figures:**
* **fig09** — Scatter: total params × total FLOPs × peak memory. 3 panels: (params, FLOPs), (params, memory), (FLOPs, memory). Color = family. Marker shape = schema (circle = standard_vflh, square = native_fused, triangle = decoupled_vision). Annotated with model names.
* **fig10** — Per-component breakdown: 4-panel grid (params, FLOPs, memory, tokens). Each panel is a stacked bar with V/F/L/H + extended native components. Schema-aware.
* **fig11** — Freezing analysis: side-by-side total-vs-trainable bars per model, log y, annotated with trainable %.

**Inputs:** `../outputs/measurements/master_wide.csv` (built by notebook 06) — required.

**Outputs:**
* `../outputs/figures/fig09_scatter_param_flops_memory.{pdf,png}`
* `../outputs/figures/fig10_per_component_breakdown.{pdf,png}`
* `../outputs/figures/fig11_freezing_analysis.{pdf,png}`
* `../outputs/figures/nb09_captions.md` — proposed captions for paper inclusion.

**Compute:** CPU only. **Runtime:** under 30 seconds.

**How to read this notebook:** run top-to-bottom. After each figure cell, inspect the rendered output — if a label collides or a color is wrong, tweak the cell, re-run, and commit the generator script change.

---
## 1. Setup (Runpod)

In [ ]:
%%capture
!pip install -q -r ../requirements.txt

In [ ]:
import os, sys, json
from pathlib import Path

NB_DIR = Path.cwd()
CODE_ACTIVE = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(CODE_ACTIVE))

MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
FIG_DIR = CODE_ACTIVE / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Paper-quality defaults
plt.rcParams.update({
    'figure.dpi':   300,
    'savefig.dpi':  300,
    'font.family':  'serif',
    'font.size':    10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
})

from vlm_audit import MODELS
from vlm_audit.components import COMPONENT_NAMES, EXTENDED_COMPONENT_NAMES
print(f'Plot defaults set. Models in roster: {len(MODELS)}')

---
## 2. Schema helpers (copied from nb02 v3)

In [ ]:
NATIVE_FUSED_FAMILIES = {'native_early_fusion'}
DECOUPLED_VISION_FAMILIES = {'native_decoupled_encoder'}

def schema_for(family: str) -> str:
    if family in NATIVE_FUSED_FAMILIES:
        return 'native_fused'
    if family in DECOUPLED_VISION_FAMILIES:
        return 'decoupled_vision'
    return 'standard_vflh'

EXTENDED_PARTS = list(EXTENDED_COMPONENT_NAMES)
VISION_LIKE_PARTS = ['vision', 'vision_und', 'vision_gen']
LANGUAGE_LIKE_PARTS = ['language', 'fused_transformer']

def safe_sum(df, parts, suffix='__params_total'):
    cols = [f'{p}{suffix}' for p in parts if f'{p}{suffix}' in df.columns]
    if not cols:
        return pd.Series([0] * len(df), index=df.index)
    return df[cols].fillna(0).sum(axis=1)

# Marker shape by schema (used in fig09)
SCHEMA_MARKER = {'standard_vflh': 'o', 'native_fused': 's', 'decoupled_vision': '^'}
# Color by family (used in fig09 and fig10)
print('Helpers OK.')

---
## 3. Load master_wide.csv + derive totals

In [ ]:
wide_path = MEASUREMENT_DIR / 'master_wide.csv'
if not wide_path.exists():
    raise FileNotFoundError(f'{wide_path} missing — run notebook 06 first.')
df = pd.read_csv(wide_path)
if 'schema' not in df.columns:
    df['schema'] = df['family'].apply(schema_for)

for part in EXTENDED_PARTS:
    for m in ['params_total', 'params_trainable', 'flops_g', 'peak_mem_mb', 'output_tokens']:
        c = f'{part}__{m}'
        if c not in df.columns:
            df[c] = np.nan

df['total_params']    = safe_sum(df, EXTENDED_PARTS, '__params_total')
df['total_flops_g']   = safe_sum(df, EXTENDED_PARTS, '__flops_g')
df['total_mem_mb']    = safe_sum(df, EXTENDED_PARTS, '__peak_mem_mb')
df['total_trainable'] = safe_sum(df, EXTENDED_PARTS, '__params_trainable')
df['trainable_pct']   = (100 * df['total_trainable'] / df['total_params'].clip(lower=1)).round(1)

# vision tokens (prefer vision_und, fall back to vision)
vision_tok = df['vision__output_tokens'].copy()
if 'vision_und__output_tokens' in df.columns:
    vision_tok = vision_tok.fillna(df['vision_und__output_tokens'])
df['vision_tokens'] = vision_tok

# Sort canonical order: family then year then name
df = df.sort_values(['family', 'release_year', 'name']).reset_index(drop=True)
print(f'Loaded {len(df)} models · families: {df["family"].nunique()} · schemas: {sorted(df["schema"].unique())}')
df[['name', 'family', 'schema', 'release_year', 'total_params', 'total_flops_g', 'total_mem_mb', 'total_trainable']].head(20)

---
## 4. Figure 09 — Triple-axis scatter (params × FLOPs × memory)

Three panels side by side. Color encodes architectural family (categorical). Marker shape encodes schema (circle / square / triangle) so native models stand out at a glance. Annotations on every dot.

Reading the figure: dots in the upper-right are large + compute-heavy; dots in the lower-left are small + cheap. Color clusters reveal that families occupy distinct regions.

In [ ]:
families = sorted(df['family'].unique())
cmap = plt.get_cmap('tab10')
fam_color = {fam: cmap(i % 10) for i, fam in enumerate(families)}

fig, axes = plt.subplots(1, 3, figsize=(18, 6.5))

# Panel A: params × FLOPs
axA = axes[0]
for fam in families:
    sub = df[(df['family'] == fam) & (df['total_params'] > 0) & (df['total_flops_g'] > 0)]
    if sub.empty:
        continue
    for _, r in sub.iterrows():
        marker = SCHEMA_MARKER.get(r['schema'], 'o')
        axA.scatter(r['total_params'], r['total_flops_g'], s=200, color=fam_color[fam],
                    marker=marker, edgecolor='black', linewidth=0.7, alpha=0.85)
        axA.annotate(r['name'], (r['total_params'], r['total_flops_g']),
                     xytext=(6, 4), textcoords='offset points', fontsize=7)
axA.set_xscale('log'); axA.set_yscale('log')
axA.set_xlabel('Total parameters')
axA.set_ylabel('Total forward FLOPs (G)')
axA.set_title('Panel A: Params × FLOPs')
axA.grid(True, which='both', linestyle='--', alpha=0.3)

# Panel B: params × memory
axB = axes[1]
for fam in families:
    sub = df[(df['family'] == fam) & (df['total_params'] > 0) & (df['total_mem_mb'] > 0)]
    if sub.empty:
        continue
    for _, r in sub.iterrows():
        marker = SCHEMA_MARKER.get(r['schema'], 'o')
        axB.scatter(r['total_params'], r['total_mem_mb'], s=200, color=fam_color[fam],
                    marker=marker, edgecolor='black', linewidth=0.7, alpha=0.85)
        axB.annotate(r['name'], (r['total_params'], r['total_mem_mb']),
                     xytext=(6, 4), textcoords='offset points', fontsize=7)
axB.set_xscale('log'); axB.set_yscale('log')
axB.set_xlabel('Total parameters')
axB.set_ylabel('Peak GPU memory (MB)')
axB.set_title('Panel B: Params × Memory')
axB.grid(True, which='both', linestyle='--', alpha=0.3)

# Panel C: FLOPs × memory
axC = axes[2]
for fam in families:
    sub = df[(df['family'] == fam) & (df['total_flops_g'] > 0) & (df['total_mem_mb'] > 0)]
    if sub.empty:
        continue
    for _, r in sub.iterrows():
        marker = SCHEMA_MARKER.get(r['schema'], 'o')
        axC.scatter(r['total_flops_g'], r['total_mem_mb'], s=200, color=fam_color[fam],
                    marker=marker, edgecolor='black', linewidth=0.7, alpha=0.85)
        axC.annotate(r['name'], (r['total_flops_g'], r['total_mem_mb']),
                     xytext=(6, 4), textcoords='offset points', fontsize=7)
axC.set_xscale('log'); axC.set_yscale('log')
axC.set_xlabel('Total forward FLOPs (G)')
axC.set_ylabel('Peak GPU memory (MB)')
axC.set_title('Panel C: FLOPs × Memory')
axC.grid(True, which='both', linestyle='--', alpha=0.3)

# Shared legend at the top of the figure: families (color) + schemas (shape)
family_handles = [Patch(facecolor=fam_color[fam], label=fam.replace('_', ' '), edgecolor='black') for fam in families]
shape_handles = [Line2D([0], [0], marker=mk, color='w', markerfacecolor='gray', markeredgecolor='black',
                         markersize=10, label=sch.replace('_', ' '))
                 for sch, mk in SCHEMA_MARKER.items()]
fig.legend(handles=family_handles, loc='upper center', ncol=min(len(families), 7),
           frameon=False, bbox_to_anchor=(0.5, 1.08), title='Family (color)')
fig.legend(handles=shape_handles, loc='upper center', ncol=3,
           frameon=False, bbox_to_anchor=(0.5, 1.00), title='Schema (marker)')
fig.suptitle(f'Architectural asymmetry across families (n={len(df)})', y=1.15)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig09_scatter_param_flops_memory.pdf', bbox_inches='tight')
plt.savefig(FIG_DIR / 'fig09_scatter_param_flops_memory.png', bbox_inches='tight')
plt.show()
print(f'Saved fig09 → {FIG_DIR}/fig09_scatter_param_flops_memory.{{pdf,png}}')

---
## 5. Figure 10 — Per-component breakdown (4-panel: params / FLOPs / memory / tokens)

Four panels in a 2×2 grid. Each panel is a stacked bar (one bar per model, components stacked by color). Standard V/F/L/H colors used everywhere; extended components (`fused_transformer`, `vision_und`, `vision_gen`, `embeddings`) get their own colors so Chameleon and Janus-Pro are visually distinct from the standard rows.

X-axis order: families clustered together, sorted by release year within family. Y-axis is log/symlog so the 100× range of total params is visible.

In [ ]:
COMP_COLORS = {
    'vision':             '#2ECC71',  # green
    'fusion':             '#E67E22',  # orange
    'language':           '#3498DB',  # blue
    'head':               '#95A5A6',  # gray
    'fused_transformer':  '#9B59B6',  # purple (Chameleon)
    'vision_und':         '#1ABC9C',  # teal (Janus understanding)
    'vision_gen':         '#16A085',  # dark teal (Janus generation)
    'embeddings':         '#F39C12',  # amber
}
STACK_ORDER = ['vision', 'vision_und', 'vision_gen', 'fusion', 'fused_transformer', 'language', 'embeddings', 'head']

fig, axes = plt.subplots(2, 2, figsize=(20, 12), sharex=True)
axes_flat = axes.flatten()
metrics = [
    ('__params_total',  'Parameters (M)',  1e6,  'symlog'),
    ('__flops_g',       'FLOPs (G)',       1.0,  'symlog'),
    ('__peak_mem_mb',   'Peak memory (MB)', 1.0, 'linear'),
    ('__output_tokens', 'Output tokens',   1.0,  'symlog'),
]
x = np.arange(len(df))
names = df['name'].tolist()

for ax, (suffix, ylabel, scale, yscale) in zip(axes_flat, metrics):
    bottoms = np.zeros(len(df))
    for comp in STACK_ORDER:
        col = f'{comp}{suffix}'
        if col not in df.columns:
            continue
        vals = (df[col].fillna(0) / scale).values
        if vals.sum() == 0:
            continue
        ax.bar(x, vals, bottom=bottoms, color=COMP_COLORS[comp], edgecolor='black', linewidth=0.4, label=comp)
        bottoms += vals
    ax.set_ylabel(ylabel)
    ax.set_yscale(yscale)
    ax.set_title(ylabel.split(' (')[0])
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)

# Shared x-tick labels at the bottom of each column.
# IMPORTANT: with sharex=True, calling set_xticklabels([]) on the top
# row would propagate empty labels to the bottom row (shared x-axis).
# Use tick_params(labelbottom=False) instead — it controls per-axis
# label *visibility* without rewriting the shared label data.
for ax in axes_flat[2:]:
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=60, ha='right', fontsize=7)
for ax in axes_flat[:2]:
    ax.tick_params(axis='x', labelbottom=False)

# Single shared legend at the top
handles = [Patch(color=COMP_COLORS[c], ec='black', label=c) for c in STACK_ORDER]
fig.legend(handles=handles, loc='upper center', ncol=len(STACK_ORDER), frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.suptitle(f'Per-component breakdown across families (n={len(df)})', y=1.06)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig10_per_component_breakdown.pdf', bbox_inches='tight')
plt.savefig(FIG_DIR / 'fig10_per_component_breakdown.png', bbox_inches='tight')
plt.show()
print(f'Saved fig10 → {FIG_DIR}/fig10_per_component_breakdown.{{pdf,png}}')

---
## 6. Figure 11 — Freezing analysis (total vs trainable params)

Side-by-side bars per model: tall gray = total, short red = trainable. Log y so the gap is visible even for small models. Each red bar annotated with `trainable_pct` so readers don't have to mentally divide. The figure visually establishes Finding F5 (the bimodal distribution).

In [ ]:
names = df['name'].tolist()
x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(16, 7))
ax.bar(x - width/2, df['total_params'] / 1e6, width=width, color='lightgray', edgecolor='black', label='Total params (M)')
ax.bar(x + width/2, df['total_trainable'] / 1e6, width=width, color='#E74C3C', edgecolor='black', label='Trainable params (M, canonical recipe)')
ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=60, ha='right', fontsize=8)
ax.set_ylabel('Parameters (M, log scale)')
ax.set_title('Total vs trainable parameters per model — visualizing the frozen-capacity asymmetry (F5)\n'
             'Tall gray + short red bars = highly frozen models (adapter-LLM, bridge-module families)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, axis='y', linestyle='--', alpha=0.3)
for i, (_, r) in enumerate(df.iterrows()):
    pct = r['trainable_pct'] if pd.notna(r['trainable_pct']) else 0
    ax.annotate(f'{pct:.0f}%', xy=(i + width/2, max(r['total_trainable'] / 1e6, 0.05)),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=7, color='#E74C3C')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig11_freezing_analysis.pdf', bbox_inches='tight')
plt.savefig(FIG_DIR / 'fig11_freezing_analysis.png', bbox_inches='tight')
plt.show()
print(f'Saved fig11 → {FIG_DIR}/fig11_freezing_analysis.{{pdf,png}}')

---
## 7. Proposed paper captions

Suggested captions for each figure, in the style expected by the venue (single paragraph each, ~3-4 sentences, refers to the panels by letter where applicable, mentions n explicitly).

In [ ]:
captions = []

captions.append('**Figure 9.** Total parameters, total forward FLOPs and peak GPU memory '
                f'across n={len(df)} vision-language models. Color encodes architectural family; marker '
                'shape encodes schema (circle = standard V/F/L/H, square = native early-fusion, '
                'triangle = decoupled-vision). Panel A (params × FLOPs) shows that families occupy '
                'distinct regions in the (P, F) plane; Panel B (params × memory) shows that the '
                'memory axis is largely driven by image-token count rather than parameter count; '
                'Panel C (FLOPs × memory) shows that the two axes are well-correlated within family '
                'but diverge across families.')

captions.append('**Figure 10.** Per-component breakdown of parameters, FLOPs, peak memory and output '
                f'tokens across n={len(df)} models. Each model is one stacked bar; the eight slots '
                'cover both the standard V/F/L/H components and the extended slots (fused_transformer '
                'for Chameleon-7B, vision_und/vision_gen for Janus-Pro-7B, embeddings for tokenizer-'
                'side components). Bars are grouped by family along the x-axis. The asymmetry pattern '
                'is visible at a glance: green-dominant bars (CLIP, SigLIP, BLIP) versus blue-dominant '
                'bars (LLaVA, IDEFICS, Idefics2).')

captions.append('**Figure 11.** Total versus trainable parameters per model (log y-axis). For each '
                f'of the n={len(df)} models, the gray bar shows total capacity and the red bar shows '
                'the parameters that are trainable under the model\'s canonical training recipe. '
                'Annotations give the trainable percentage. Adapter-LLM and bridge-module models '
                '(BLIP-2, InstructBLIP, LLaVA, IDEFICS, Idefics2, Pixtral, DeepSeek-VL2) show '
                'dramatic gaps (1-10% trainable); contrastive and single-stream models (CLIP, SigLIP, '
                'ViLT) train end-to-end (~100% trainable). The bimodal distribution supports Finding F5.')

cap_path = FIG_DIR / 'nb09_captions.md'
cap_path.write_text('\n\n'.join(captions), encoding='utf-8')
print(f'Saved captions → {cap_path}')
for c in captions:
    print(c)
    print()

---
## 8. Output inventory

In [ ]:
outputs = [
    ('fig09_scatter_param_flops_memory.pdf', FIG_DIR / 'fig09_scatter_param_flops_memory.pdf'),
    ('fig09_scatter_param_flops_memory.png', FIG_DIR / 'fig09_scatter_param_flops_memory.png'),
    ('fig10_per_component_breakdown.pdf',    FIG_DIR / 'fig10_per_component_breakdown.pdf'),
    ('fig10_per_component_breakdown.png',    FIG_DIR / 'fig10_per_component_breakdown.png'),
    ('fig11_freezing_analysis.pdf',          FIG_DIR / 'fig11_freezing_analysis.pdf'),
    ('fig11_freezing_analysis.png',          FIG_DIR / 'fig11_freezing_analysis.png'),
    ('nb09_captions.md',                     FIG_DIR / 'nb09_captions.md'),
]
print(f'{"FILE":42s}  EXISTS  SIZE')
print('-' * 70)
for label, p in outputs:
    if p.exists():
        sz_kb = p.stat().st_size / 1024
        print(f'{label:42s}  ✓       {sz_kb:>8.1f} KB')
    else:
        print(f'{label:42s}  ✗       (missing)')

---
## Next steps

* PDFs are paper-ready — drop them into the manuscript's `figures/` directory.
* PNGs are slide-ready — use them for presentations and supplementary material.
* Captions in `nb09_captions.md` are starting points; tighten them to fit the venue's caption-length limit.
* If any figure needs a different cut (e.g. only contrastive vs adapter, or by-year facets), branch this notebook into a `09b_alt_figures.ipynb` and reuse the styling defaults from §1.

*End of notebook 09 (v3 — 2026-05-22).*